# IndicTrans2 HF Inference

We provide an example notebook on how to use our IndicTrans2 models which were originally trained with the fairseq to HuggingFace transformers for inference purpose.


## Setup

Please run the cells below to install the necessary dependencies.


In [6]:
!pip install --force-reinstall "transformers==4.46.3"

  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached filelock-4.2.10-py3-none-any.whl.metadata (2.3 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 109, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 102, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 106, in wrapper
    return func(self, options, args)
  File

In [6]:
import os
import sys
import transformers

print("Transformers:", transformers.__version__)
print("Python:", sys.version)
print("Current directory:", os.getcwd())
print("Toolkit folder exists:", os.path.exists("/content/IndicTransToolkit"))
print("Toolkit package exists:", os.path.exists("/content/IndicTransToolkit/IndicTransToolkit"))

Transformers: 4.53.2
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Current directory: /content/IndicTrans2/huggingface_interface
Toolkit folder exists: True
Toolkit package exists: True


In [7]:
!pip show indictranstoolkit

Name: indictranstoolkit
Version: 1.1.1
Summary: A simple, consistent, and extendable module for IndicTrans2 compatible with HF models
Home-page: 
Author: 
Author-email: Varun Gumma <varun230999@gmail.com>
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Editable project location: /content/IndicTrans2/huggingface_interface/IndicTransToolkit
Requires: cython, indic-nlp-library-itt, sacrebleu, sacremoses, transformers
Required-by: 


In [8]:
import IndicTransToolkit
import os

print("Package location:", IndicTransToolkit.__file__)
print("Package paths:", list(IndicTransToolkit.__path__))

package_path = list(IndicTransToolkit.__path__)[0]
print("Files inside package:", os.listdir(package_path))


Package location: None
Package paths: ['/content/IndicTransToolkit', '/content/IndicTrans2/huggingface_interface/IndicTransToolkit']
Files inside package: ['CHANGELOG.md', 'IndicTransToolkit', '.gitignore', 'LICENSE', 'indictranstoolkit.egg-info', 'setup.py', '.git', '.github', 'pyproject.toml', 'MANIFEST.in', 'README.md']


In [9]:
import sys

# Remove the duplicate toolkit path from Python's search path
sys.path = [
    p for p in sys.path
    if p != "/content/IndicTransToolkit"
]

# Prioritize the toolkit installed with IndicTrans2
sys.path.insert(
    0,
    "/content/IndicTrans2/huggingface_interface"
)

import IndicTransToolkit
print("Package location:", list(IndicTransToolkit.__path__))

Package location: ['/content/IndicTrans2/huggingface_interface/IndicTransToolkit', '/content/IndicTransToolkit', '/content/IndicTrans2/huggingface_interface/IndicTransToolkit']


In [10]:
from IndicTransToolkit.processor import IndicProcessor

print("IndicProcessor imported successfully!")

IndicProcessor imported successfully!


In [1]:
%%capture
!git clone https://github.com/AI4Bharat/IndicTrans2.git

In [2]:
%%capture
%cd /content/IndicTrans2/huggingface_interface

In [3]:
%%capture
!python3 -m pip install nltk sacremoses pandas regex mock transformers==4.53.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

In [4]:
from huggingface_hub import login
from getpass import getpass

token = getpass("Hugging Face token: ")
login(token=token)

print("Login successful!")

Hugging Face token: ··········
Login successful!


**IMPORTANT : Restart your run-time first and then run the cells below.**

## Inference


In [13]:
import sys
import torch

# Prioritize the correct IndicTrans2 toolkit
sys.path = [
    p for p in sys.path
    if p != "/content/IndicTransToolkit"
]

sys.path.insert(
    0,
    "/content/IndicTrans2/huggingface_interface"
)

from transformers import (
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    AutoTokenizer
)

from IndicTransToolkit.processor import IndicProcessor

BATCH_SIZE = 4
DEVICE = "cpu"
quantization = None

print("All imports successful!")
print("Device:", DEVICE)

All imports successful!
Device: cpu


In [2]:
%cd /content/IndicTrans2/huggingface_interface

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
!pip install -e ./IndicTransToolkit

[Errno 2] No such file or directory: '/content/IndicTrans2/huggingface_interface'
/content
Cloning into 'IndicTransToolkit'...
remote: Enumerating objects: 288, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 288 (delta 89), reused 76 (delta 62), pack-reused 155 (from 1)
Receiving objects: 100% (288/288), 4.41 MiB | 14.14 MiB/s, done.
Resolving deltas: 100% (128/128), done.
Obtaining file:///content/IndicTransToolkit
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 6.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 20.4 MB/s  0:00:00
  Building editable for indictranstoolkit (pyproject.toml) ... done
  Created wheel for indictranstoolkit: filename=indictranstoolkit-1.1.1-0.editable-

In [14]:
def initialize_model_and_tokenizer(ckpt_dir, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig == None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        # Preprocess the batch and extract entity mappings
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        # Tokenize the batch and generate input encodings
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations using the model
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode the generated tokens into text
        generated_tokens = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        # Postprocess the translations, including entity replacement
        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations

### English to Indic Example


In [ ]:
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"  # ai4bharat/indictrans2-en-indic-dist-200M
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, quantization)

ip = IndicProcessor(inference=True)

en_sents = [
    "When I was young, I used to go to the park every day.",
    "He has many old books, which he inherited from his ancestors.",
    "I can't figure out how to solve my problem.",
    "She is very hardworking and intelligent, which is why she got all the good marks.",
    "We watched a new movie last week, which was very inspiring.",
    "If you had met me at that time, we would have gone out to eat.",
    "She went to the market with her sister to buy a new sari.",
    "Raj told me that he is going to his grandmother's house next month.",
    "All the kids were having fun at the party and were eating lots of sweets.",
    "My friend has invited me to his birthday party, and I will give him a gift.",
]

src_lang, tgt_lang = "eng_Latn", "hin_Deva"
hi_translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(en_sents, hi_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")

# flush the models to free the GPU memory
del en_indic_tokenizer, en_indic_model


eng_Latn - hin_Deva
eng_Latn: When I was young, I used to go to the park every day.
hin_Deva: जब मैं छोटा था, मैं हर दिन पार्क जाता था।
eng_Latn: He has many old books, which he inherited from his ancestors.
hin_Deva: उनके पास कई पुरानी किताबें हैं, जो उन्हें अपने पूर्वजों से विरासत में मिली हैं।
eng_Latn: I can't figure out how to solve my problem.
hin_Deva: मुझे समझ नहीं आ रहा है कि मैं अपनी समस्या का समाधान कैसे करूं।
eng_Latn: She is very hardworking and intelligent, which is why she got all the good marks.
hin_Deva: वह बहुत मेहनती और बुद्धिमान है, यही कारण है कि उसे सभी अच्छे अंक मिले।
eng_Latn: We watched a new movie last week, which was very inspiring.
hin_Deva: हमने पिछले हफ्ते एक नई फिल्म देखी, जो बहुत प्रेरणादायक थी।
eng_Latn: If you had met me at that time, we would have gone out to eat.
hin_Deva: अगर आप उस समय मुझसे मिलते तो हम बाहर खाना खाने जाते।
eng_Latn: She went to the market with her sister to buy a new sari.
hin_Deva: वह अपनी बहन के साथ नई साड़ी खरीदने के लिए बाजार 

### Indic to English Example

In [15]:
indic_en_ckpt_dir = "ai4bharat/indictrans2-indic-en-dist-200M"  # ai4bharat/indictrans2-indic-en-dist-200M
indic_en_tokenizer, indic_en_model = initialize_model_and_tokenizer(indic_en_ckpt_dir, quantization)

ip = IndicProcessor(inference=True)

hi_sents = [
    "जब मैं छोटा था, मैं हर रोज़ पार्क जाता था।",
    "उसके पास बहुत सारी पुरानी किताबें हैं, जिन्हें उसने अपने दादा-परदादा से विरासत में पाया।",
    "मुझे समझ में नहीं आ रहा कि मैं अपनी समस्या का समाधान कैसे ढूंढूं।",
    "वह बहुत मेहनती और समझदार है, इसलिए उसे सभी अच्छे मार्क्स मिले।",
    "हमने पिछले सप्ताह एक नई फिल्म देखी जो कि बहुत प्रेरणादायक थी।",
    "अगर तुम मुझे उस समय पास मिलते, तो हम बाहर खाना खाने चलते।",
    "वह अपनी दीदी के साथ बाजार गयी थी ताकि वह नई साड़ी खरीद सके।",
    "राज ने मुझसे कहा कि वह अगले महीने अपनी नानी के घर जा रहा है।",
    "सभी बच्चे पार्टी में मज़ा कर रहे थे और खूब सारी मिठाइयाँ खा रहे थे।",
    "मेरे मित्र ने मुझे उसके जन्मदिन की पार्टी में बुलाया है, और मैं उसे एक तोहफा दूंगा।",
]
src_lang, tgt_lang = "hin_Deva", "eng_Latn"
en_translations = batch_translate(hi_sents, src_lang, tgt_lang, indic_en_model, indic_en_tokenizer, ip)


print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(hi_sents, en_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")

# flush the models to free the GPU memory
del indic_en_tokenizer, indic_en_model

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]


hin_Deva - eng_Latn
hin_Deva: जब मैं छोटा था, मैं हर रोज़ पार्क जाता था।
eng_Latn: When I was young, I used to go to the park every day.
hin_Deva: उसके पास बहुत सारी पुरानी किताबें हैं, जिन्हें उसने अपने दादा-परदादा से विरासत में पाया।
eng_Latn: He has a lot of old books, which he inherited from his grandparents.
hin_Deva: मुझे समझ में नहीं आ रहा कि मैं अपनी समस्या का समाधान कैसे ढूंढूं।
eng_Latn: I can't figure out how to find a solution to my problem.
hin_Deva: वह बहुत मेहनती और समझदार है, इसलिए उसे सभी अच्छे मार्क्स मिले।
eng_Latn: He is very hardworking and understanding, so he got all the good marks.
hin_Deva: हमने पिछले सप्ताह एक नई फिल्म देखी जो कि बहुत प्रेरणादायक थी।
eng_Latn: We saw a new film last week which was very inspiring.
hin_Deva: अगर तुम मुझे उस समय पास मिलते, तो हम बाहर खाना खाने चलते।
eng_Latn: If you found me nearby at that time, we would go out to eat.
hin_Deva: वह अपनी दीदी के साथ बाजार गयी थी ताकि वह नई साड़ी खरीद सके।
eng_Latn: She had gone to the market with

In [18]:
print("Model loaded:", "indic_en_model" in globals())
print("Tokenizer loaded:", "indic_en_tokenizer" in globals())
print("Processor loaded:", "ip" in globals())

Model loaded: True
Tokenizer loaded: True
Processor loaded: True


In [19]:

import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/grouped_data.zip")
INPUT_DIR = Path("/content/grouped_input")
OUTPUT_DIR = Path("/content/translated")

INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(INPUT_DIR)

files = sorted(INPUT_DIR.rglob("sample-*.txt"))
print("Files found:", len(files))

for file_path in files:
    print(f"\nTranslating {file_path.name}...", flush=True)

    lines = [
        line.strip()
        for line in file_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    output = []
    current_group = []
    group_number = 0

    def translate_group(sentences, number):
        translations = batch_translate(
            sentences,
            "hin_Deva",
            "eng_Latn",
            indic_en_model,
            indic_en_tokenizer,
            ip
        )

        output.append(f"GROUP {number}")
        output.extend(translations)
        output.append("")

    for line in lines:
        if line.upper().startswith("GROUP"):
            if current_group:
                group_number += 1
                translate_group(current_group, group_number)
                current_group = []
        else:
            current_group.append(line)

    if current_group:
        group_number += 1
        translate_group(current_group, group_number)

    output_path = OUTPUT_DIR / file_path.name
    output_path.write_text("\n".join(output), encoding="utf-8")

    print(f"Completed: {file_path.name} | Groups: {group_number}", flush=True)

print("\nALL TRANSLATIONS COMPLETED!")


Files found: 7

Translating sample-1.txt...
Completed: sample-1.txt | Groups: 10

Translating sample-2.txt...
Completed: sample-2.txt | Groups: 11

Translating sample-3.txt...
Completed: sample-3.txt | Groups: 10

Translating sample-4.txt...
Completed: sample-4.txt | Groups: 10

Translating sample-5.txt...
Completed: sample-5.txt | Groups: 11

Translating sample-6.txt...
Completed: sample-6.txt | Groups: 11

Translating sample-7.txt...
Completed: sample-7.txt | Groups: 13

ALL TRANSLATIONS COMPLETED!


In [20]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/translated_results",
    "zip",
    "/content/translated"
)

files.download("/content/translated_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
indic_en_ckpt_dir = "ai4bharat/indictrans2-indic-en-dist-200M"

indic_en_tokenizer, indic_en_model = initialize_model_and_tokenizer(
    indic_en_ckpt_dir,
    quantization
)

print("Model and tokenizer loaded successfully!")

Model and tokenizer loaded successfully!


In [5]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Paste your Hugging Face token: ")
login(token=hf_token)

print("Hugging Face login successful!")

Paste your Hugging Face token: ··········
Hugging Face login successful!


### Indic to Indic Example


In [ ]:
indic_indic_ckpt_dir = "ai4bharat/indictrans2-indic-indic-1B"  # ai4bharat/indictrans2-indic-indic-dist-320M
indic_indic_tokenizer, indic_indic_model = initialize_model_and_tokenizer(indic_indic_ckpt_dir, quantization)

ip = IndicProcessor(inference=True)

hi_sents = [
    "जब मैं छोटा था, मैं हर रोज़ पार्क जाता था।",
    "उसके पास बहुत सारी पुरानी किताबें हैं, जिन्हें उसने अपने दादा-परदादा से विरासत में पाया।",
    "मुझे समझ में नहीं आ रहा कि मैं अपनी समस्या का समाधान कैसे ढूंढूं।",
    "वह बहुत मेहनती और समझदार है, इसलिए उसे सभी अच्छे मार्क्स मिले।",
    "हमने पिछले सप्ताह एक नई फिल्म देखी जो कि बहुत प्रेरणादायक थी।",
    "अगर तुम मुझे उस समय पास मिलते, तो हम बाहर खाना खाने चलते।",
    "वह अपनी दीदी के साथ बाजार गयी थी ताकि वह नई साड़ी खरीद सके।",
    "राज ने मुझसे कहा कि वह अगले महीने अपनी नानी के घर जा रहा है।",
    "सभी बच्चे पार्टी में मज़ा कर रहे थे और खूब सारी मिठाइयाँ खा रहे थे।",
    "मेरे मित्र ने मुझे उसके जन्मदिन की पार्टी में बुलाया है, और मैं उसे एक तोहफा दूंगा।",
]
src_lang, tgt_lang = "hin_Deva", "mar_Deva"
mr_translations = batch_translate(hi_sents, src_lang, tgt_lang, indic_indic_model, indic_indic_tokenizer, ip)

print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(hi_sents, mr_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")

# flush the models to free the GPU memory
del indic_indic_tokenizer, indic_indic_model


hin_Deva - mar_Deva
hin_Deva: जब मैं छोटा था, मैं हर रोज़ पार्क जाता था।
mar_Deva: मी लहान होतो तेव्हा मी दररोज उद्यानाला जायचे.
hin_Deva: उसके पास बहुत सारी पुरानी किताबें हैं, जिन्हें उसने अपने दादा-परदादा से विरासत में पाया।
mar_Deva: तिच्याकडे बरीच जुनी पुस्तके आहेत, जी तिला तिच्या आजोबांकडून वारशाने मिळाली आहेत.
hin_Deva: मुझे समझ में नहीं आ रहा कि मैं अपनी समस्या का समाधान कैसे ढूंढूं।
mar_Deva: माझ्या समस्येवर तोडगा कसा काढायचा हे मला समजत नाही.
hin_Deva: वह बहुत मेहनती और समझदार है, इसलिए उसे सभी अच्छे मार्क्स मिले।
mar_Deva: तो खूप मेहनती आणि बुद्धिमान आहे, त्यामुळे त्याला सर्व चांगले गुण मिळाले.
hin_Deva: हमने पिछले सप्ताह एक नई फिल्म देखी जो कि बहुत प्रेरणादायक थी।
mar_Deva: आम्ही गेल्या आठवड्यात एक नवीन चित्रपट पाहिला जो खूप प्रेरणादायी होता.
hin_Deva: अगर तुम मुझे उस समय पास मिलते, तो हम बाहर खाना खाने चलते।
mar_Deva: जर तुम्हाला त्या वेळी मला पास मिळाला तर आम्ही बाहेर जेवायला जाऊ.
hin_Deva: वह अपनी दीदी के साथ बाजार गयी थी ताकि वह नई साड़ी खरीद सके।
mar_Deva: ती तिच्या ब